# Notebook 03 — RAG Pipeline sobre Corpus Etiquetado

**Curso:** Minería de Textos | **Proyecto 3** | CUC

---

## IMPORTANTE: Este notebook se ejecuta DESPUÉS del Notebook 02

El RAG se construye sobre el corpus ya etiquetado por el clasificador fine-tuneado.
Cada chunk hereda la emoción asignada por el modelo, lo que permite filtrado preciso.

```
Notebook 02 → Fine-Tuning → etiquetar_corpus_con_modelo() → corpus_con_emociones.pkl
                                        ↓
Notebook 03 → Chunking + Embeddings + FAISS → RAG listo
```

In [ ]:
!pip install -q pymongo[srv] sentence-transformers faiss-cpu transformers python-dotenv numpy

In [1]:
import sys, os
sys.path.append('../app')
sys.path.append('../src')

import numpy as np
import json
from pathlib import Path

from app.config import  TOP_K, FLAN_T5_MODEL, SYSTEM_PROMPT,CACHE_EMBEDDINGS, CACHE_CHUNKS, CACHE_FAISS
from src.mongo_utils import mongo_utils
from src.finetuning_utils import finetuning_utils
from src.rag_utils import rag_utils
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
print('Librerias cargadas.')

# Limpiando cache
for f in [CACHE_EMBEDDINGS, CACHE_CHUNKS, CACHE_FAISS]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Eliminado: {f}")
    else:
        print(f"No existe (ya estaba limpio): {f}")

C:\Users\pmari\OneDrive\Para Revisar\Documentos\Pablo\Cuc\2026\Mineria de Textos\chatbot-musical-inteligente\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Librerias cargadas.
Eliminado: C:\Users\pmari\OneDrive\Para Revisar\Documentos\GitHub\chatbot-musical-inteligente\data\embeddings_cache\embeddings_parrafos.npy
Eliminado: C:\Users\pmari\OneDrive\Para Revisar\Documentos\GitHub\chatbot-musical-inteligente\data\embeddings_cache\chunks_parrafos.pkl
Eliminado: C:\Users\pmari\OneDrive\Para Revisar\Documentos\GitHub\chatbot-musical-inteligente\data\embeddings_cache\indice_faiss.index


## 1. Cargar corpus etiquetado por el modelo fine-tuneado

In [2]:
db_conexion = mongo_utils()
finetuning = finetuning_utils()

if db_conexion.verificar_conexion():
   canciones_raw=db_conexion.cargar_canciones()
   canciones_etiquetadas = finetuning.etiquetar_corpus_con_modelo(canciones_raw)
   print(f'Canciones con emocion: {len(canciones_etiquetadas)}')
   # Verificar que tienen el campo emocion
   ejemplo = canciones_etiquetadas[0]
   print(f'Ejemplo: {ejemplo.get("titulo")} — Emocion: {ejemplo.get("emocion")} ({ejemplo.get("emocion_score",0):.0%})')
else:
   canciones_raw=[]
   print(f'Canciones no cargadas: {len(canciones_raw)}')



2026-04-22 23:57:00 | INFO     | mongo_utils | Conectando a MongoDB Atlas | DB: analisisMusical | Col: analisisMusical
2026-04-22 23:57:01 | INFO     | mongo_utils | Conexión verificada correctamente.
2026-04-22 23:57:01 | INFO     | mongo_utils | Cargando canciones | limite=None | solo_con_letra=True
2026-04-22 23:57:03 | INFO     | mongo_utils | Canciones cargadas: 6940
2026-04-22 23:57:03 | INFO     | EmotionClassifier | Cargando corpus etiquetado desde caché...
2026-04-22 23:57:03 | INFO     | EmotionClassifier | 6526 canciones cargadas con emoción.
Canciones con emocion: 6526
Ejemplo: 7 rings — Emocion: rabia (89%)


## 2. Comparación de estrategias de chunking

In [3]:
rag = rag_utils()
chunks_parrafos  = []
chunks_completos = []
for c in canciones_etiquetadas:
    chunks_parrafos.extend(rag.chunking_por_parrafos(c))
    chunks_completos.extend(rag.chunking_cancion_completa(c))

print('COMPARACION DE ESTRATEGIAS DE CHUNKING')
print('=' * 60)
for nombre, lista in [('Por parrafos/estrofas', chunks_parrafos),
                       ('Cancion completa',      chunks_completos)]:
    tamanos = [len(c['texto']) for c in lista]
    print(f'\n{nombre}:')
    print(f'  Total chunks:    {len(lista)}')
    print(f'  Promedio chars:  {np.mean(tamanos):.0f}')
    print(f'  Min / Max:       {min(tamanos)} / {max(tamanos)}')
    print(f'  Ejemplo texto:   "{lista[0]["texto"][:80]}..."')
    print(f'  Emocion:         {lista[0]["emocion"]} ({lista[0]["emocion_score"]:.0%})')
    print(f'  Metadatos:       {lista[0]["titulo"]} - {lista[0]["artista"]} ({lista[0]["genero"]})')

COMPARACION DE ESTRATEGIAS DE CHUNKING

Por parrafos/estrofas:
  Total chunks:    70264
  Promedio chars:  236
  Min / Max:       31 / 586
  Ejemplo texto:   "yeah breakfast at tiffany's and bottles of bubbles girls with tattoos who like g..."
  Emocion:         rabia (89%)
  Metadatos:       7 rings - Ariana Grande (pop)

Cancion completa:
  Total chunks:    6526
  Promedio chars:  483
  Min / Max:       92 / 947
  Ejemplo texto:   "yeah breakfast at tiffany's and bottles of bubbles girls with tattoos who like g..."
  Emocion:         rabia (89%)
  Metadatos:       7 rings - Ariana Grande (pop)


## 3. Construir indice FAISS sobre corpus etiquetado

In [4]:
indice, chunks_activos = rag.inicializar(canciones_etiquetadas)
print(f'\nChunks indexados: {len(chunks_activos)}')
print(f'Ejemplo de chunk con emocion:')
ej = chunks_activos[0]
print(f'  Titulo:  {ej["titulo"]}')
print(f'  Emocion: {ej["emocion"]} ({ej["emocion_score"]:.0%})')
print(f'  Texto:   {ej["texto"][:100]}...')

2026-04-23 00:00:06 | INFO     | rag_utils | Construyendo RAG desde corpus etiquetado...
2026-04-23 00:00:06 | INFO     | rag_utils | Construyendo chunks para 6526 canciones...
2026-04-23 00:00:06 | INFO     | rag_utils | Chunks generados: 70264
2026-04-23 00:00:06 | INFO     | rag_utils | Generando embeddings para 70264 chunks...
2026-04-23 00:00:06 | INFO     | rag_utils | Cargando modelo de embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2526.43it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-04-23 00:00:14 | INFO     | rag_utils | Modelo cargado. Dimensión: 384


C:\Users\pmari\OneDrive\Para Revisar\Documentos\GitHub\chatbot-musical-inteligente\src\rag_utils.py:172: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = self._modelo_emb.get_sentence_embedding_dimension()
Batches: 100%|██████████| 1098/1098 [06:52<00:00,  2.66it/s]


2026-04-23 00:07:08 | INFO     | rag_utils | Embeddings guardados: (70264, 384)
2026-04-23 00:07:08 | INFO     | rag_utils | Construyendo índice FAISS...
2026-04-23 00:07:09 | INFO     | rag_utils | Índice guardado: 70264 vectores
2026-04-23 00:07:09 | INFO     | rag_utils | RAG listo: 70264 chunks indexados.

Chunks indexados: 70264
Ejemplo de chunk con emocion:
  Titulo:  7 rings
  Emocion: rabia (89%)
  Texto:   yeah breakfast at tiffany's and bottles of bubbles girls with tattoos who like getting in trouble la...


## 4. Pruebas de busqueda semantica con filtro por emocion

In [5]:
pruebas = [
    ('canciones sobre tristeza y desamor', 'tristeza'),
    ('musica alegre para una fiesta',      'alegria'),
    ('letras de amor romantico',           'amor'),
    ('canciones de rabia y traicion',      'rabia'),
    ('recuerdos del pasado nostalgia',     'nostalgia'),
]

for pregunta, emocion_filtro in pruebas:
    print(f'\nPregunta: {pregunta}')
    print(f'Filtro emoción: {emocion_filtro}')
    print('-' * 50)
    resultados = rag.buscar(pregunta, top_k=3, filtro_emocion=emocion_filtro)
    if not resultados:
        print('  Sin resultados con filtro. Buscando sin filtro...')
        resultados = rag.buscar(pregunta, top_k=3)
    for i, r in enumerate(resultados):
        print(f'  [{i+1}] Score: {r["score"]:.4f} | {r["emocion"]:10s} | {r["titulo"]} - {r["artista"]}')
        print(f'       {r["texto"][:80]}...')


Pregunta: canciones sobre tristeza y desamor
Filtro emoción: tristeza
--------------------------------------------------
  [1] Score: 0.4372 | tristeza   | So Sick - Justin Bieber
       thoughts of you and your memory and how every song reminds me of what used to be...
  [2] Score: 0.3212 | tristeza   | So Sick - Justin Bieber
       more walking round with my head down i'm so over being blue crying over you hook...
  [3] Score: 0.2918 | tristeza   | Sad Serenade - Selena Gomez
       a sad sad serenade sad serenade for every broken heart tonight all the love that...

Pregunta: musica alegre para una fiesta
Filtro emoción: alegria
--------------------------------------------------
  [1] Score: 0.3570 | alegria    | Dionysus - Japanese ver. - BTS (방탄소년단)
       when the night comes tumble tumble tumble studioのbass ジョウム ジョウム ジョウム bass drum g...
  [2] Score: 0.3489 | alegria    | Don’t Stop the Music (The Wideboys Club Mix) - Rihanna
       baby i'ma say your aura is incredible if you d

## 5. Generador Flan-T5 local

In [9]:

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Dispositivo: {dispositivo.upper()}')
print(f'Cargando {FLAN_T5_MODEL}...')

flan_tokenizer = AutoTokenizer.from_pretrained(FLAN_T5_MODEL)
flan_model     = AutoModelForSeq2SeqLM.from_pretrained(FLAN_T5_MODEL).to(dispositivo)

def generador(prompt_data: dict, max_new_tokens=200) -> str:
    """
    Estrategia adaptada a las capacidades de Flan-T5:
    1. Construye una respuesta estructurada en inglés
    2. La traduce al español con Flan-T5
    """
    chunks    = prompt_data.get("chunks", [])
    pregunta  = prompt_data.get("pregunta", "")
    historial = prompt_data.get("historial", "")

    # Paso 1: construir respuesta base en inglés con los chunks
    if chunks:
        canciones_str = "\n".join([
            f'- "{c["titulo"]}" by {c["artista"]} '
            f'(emotion: {c["emocion"]}, genre: {c["genero"]}): '
            f'{c["texto"][:100]}'
            for c in chunks[:3]
        ])

        prompt_ingles = (
            f"Based on these songs:\n{canciones_str}\n\n"
            f"Answer this music question briefly: {pregunta}\n"
            f"Mention song titles and artists.\n"
            f"Answer:"
        )
    else:
        prompt_ingles = (
            f"Answer briefly: {pregunta}\n"
            f"Answer:"
        )

    # Paso 2: generar respuesta en inglés
    inputs = flan_tokenizer(
        prompt_ingles,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(dispositivo)

    with torch.no_grad():
        outputs = flan_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True,
            repetition_penalty=2.0,
        )
    respuesta_ingles = flan_tokenizer.decode(
        outputs[0], skip_special_tokens=True
    ).strip()

    # Paso 3: traducir al español
    prompt_traduccion = (
        f"Translate to Spanish: {respuesta_ingles}\n"
        f"Translation:"
    )
    inputs_trad = flan_tokenizer(
        prompt_traduccion,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(dispositivo)

    with torch.no_grad():
        outputs_trad = flan_model.generate(
            **inputs_trad,
            max_new_tokens=200,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    respuesta_es = flan_tokenizer.decode(
        outputs_trad[0], skip_special_tokens=True
    ).strip()

    return respuesta_es if respuesta_es else respuesta_ingles

print('Flan-T5 listo.')

# Prueba con chunks reales
chunks_prueba = rag.buscar("canciones tristes de desamor", top_k=3, filtro_emocion="tristeza")
resultado = generador({
    "pregunta": "What sad songs about heartbreak do you recommend?",
    "chunks":   chunks_prueba,
})
print(f"Respuesta: {resultado}")

Dispositivo: CUDA
Cargando google/flan-t5-large...


Loading weights: 100%|██████████| 558/558 [00:00<00:00, 1590.66it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Flan-T5 listo.
Respuesta: "Serenade triste" de Selena Gomez (emoción: tristeza, género: pop)


## 6. Comparacion CON vs SIN RAG

In [10]:
def rag_completo_nb(pregunta, top_k=3, filtro_emocion=None):
    chunks = rag.buscar(pregunta, top_k=top_k, filtro_emocion=filtro_emocion)
    if not chunks:
        chunks = rag.buscar(pregunta, top_k=top_k)

    respuesta = generador({
        "pregunta": pregunta,
        "chunks":   chunks,
    })
    return respuesta, chunks

def sin_rag(pregunta):
    respuesta = generador({
        "pregunta": pregunta,
        "chunks":   [],   # sin contexto del corpus
    })
    return respuesta

# Comparación CON vs SIN RAG
preguntas_comp = [
    ('Que canciones hablan de desamor?',             'tristeza'),
    ('Que cancion recomiendas para alguien triste?', 'tristeza'),
    ('Dame una cancion alegre para celebrar.',        'alegria'),
]

resultados_comp = []
for pregunta, emocion in preguntas_comp:
    print(f'\n{"="*60}')
    print(f'PREGUNTA: {pregunta}')
    resp_sin         = sin_rag(pregunta)
    resp_con, chunks = rag_completo_nb(pregunta, top_k=3, filtro_emocion=emocion)
    print(f'SIN RAG: {resp_sin}')
    print(f'CON RAG: {resp_con}')
    print(f'Fuentes: {[c["titulo"]+" ("+c["emocion"]+")" for c in chunks]}')
    resultados_comp.append({
        'pregunta': pregunta,
        'sin_rag':  resp_sin,
        'con_rag':  resp_con,
        'fuentes':  [{'titulo': c['titulo'], 'artista': c['artista'],
                      'emocion': c['emocion'], 'score': c['score']}
                     for c in chunks]
    })

Path('../resultados').mkdir(exist_ok=True)
with open('../resultados/comparacion_rag.json', 'w', encoding='utf-8') as f:
    json.dump(resultados_comp, f, ensure_ascii=False, indent=2)
print('\nResultados guardados.')


PREGUNTA: Que canciones hablan de desamor?
SIN RAG: Desamar
CON RAG: "So Sick" de Justin Bieber (emoción: tristeza, género: pop): pensamientos sobre usted y su memoria y cómo cada escuela me recuerda de lo que era esa era la razón h
Fuentes: ['So Sick (tristeza)', 'Always Remember Us This Way (tristeza)']

PREGUNTA: Que cancion recomiendas para alguien triste?
SIN RAG: Canción Recomendaciones para Alguien triste
CON RAG: "La chica hecho (Remix de Catalizador)" de Beyoncé
Fuentes: ['Sad Serenade (tristeza)', 'So Sick (tristeza)', 'Broken-Hearted Girl (Catalyst Remix) (tristeza)']

PREGUNTA: Dame una cancion alegre para celebrar.
SIN RAG: Dame una canción alegre para celebrar.
CON RAG: "Birthday (Octava Disco House Radio Edit)" de Katy Perry (emoción: alegra, género: pop)
Fuentes: ['Birthday (Octava Disco House Radio Edit) (alegria)', 'Every Day Is A Holiday (alegria)', 'Birthday (alegria)']

Resultados guardados.
